In [5]:
import os
import json

import numpy as np
import pandas as pd

In [6]:
runs = [
    os.path.join('experiments', str(x)) for x in range(744, 824)
] # without +/- fix in Z_0 calculation

# runs = [
#     os.path.join('experiments', str(x)) for x in range(1154, 1194)
# ] # fixed +/- in Z_0 calculation

assert len(runs) == 80, len(runs)

# assert len(runs) == 40, len(runs)

In [7]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [8]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error',
        'time_to_1p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))
    # assert info['use_cpu'] == False
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] == min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()

    try:
        first_1p_epoch = int(log[log['rel_l2_error'] < 0.01].iloc[0]['epoch'])
        info['time_to_1p_rel_l2_error'] = log[log['epoch'] <= first_1p_epoch]['training_time'].sum().item()
    except:
        print(run, log['rel_l2_error'].min())
        continue
    
    df.loc[len(df)] = info

In [9]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_1p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_1p_rel_l2_error': 'Time to <1% RL2E. (s)',
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to <5% RL2E. (s) Time to <1% RL2E. (s)  \
Solution        w/ BC                                               
Hopf fibration  False             0.8 ± 0.0             2.7 ± 0.1   
                True              2.0 ± 0.3             5.2 ± 1.6   
Plane waves     False             3.2 ± 0.1             5.6 ± 0.2   
                True              4.0 ± 0.9            10.5 ± 0.6   
Radial waves    False             2.8 ± 0.1             7.9 ± 0.5   
                True              4.3 ± 0.2            22.2 ± 2.2   
Random solution False             1.7 ± 0.1             2.9 ± 0.1   
                True              5.3 ± 0.5            37.8 ± 4.3   

                      Time to min. RL2E. (s)       Min. RL2E. (%)  
Solution        w/ BC                                              
Hopf fibration  False           147.3 ± 11.0  3.07e-05 ± 1.67e-06  
                True             148.2 ± 9.8  5.53e-04 ± 2.09e-04  
Plane waves     False            131.9 ± 2.3  1.11e-04 ± 3.64e-06  
                True             140.6 ± 5.7  1.32e-04 ± 1.72e-05  
Radial waves    False            132.2 ± 4.3  4.10e-04 ± 2.84e-05  
                True             135.6 ± 1.9  3.25e-03 ± 5.53e-04  
Random solution False            126.6 ± 5.9  1.42e-04 ± 3.19e-06  
                True             119.1 ± 4.2  2.71e-03 ± 3.03e-04

In [13]:
info

{'input_dim': 3,
 'output_dim': 6,
 'max_epochs': 10000,
 'log_freq': 100,
 'verbose': False,
 'tol': 0.0001,
 'atol': 0.0001,
 'k': 10,
 'model_class': 'MaxwellSimple',
 'norm_inputs': False,
 'restrict_time': True,
 'data_from_lattice': False,
 'n_val': 10000,
 'init': 'kaiming',
 'noise_scale': 0,
 'inner_lr': 0.01,
 'inner_wd': 5e-06,
 'outer_lr': 0.01,
 'outer_wd': 5e-05,
 'bilevel': False,
 'symlog': False,
 'early_stopping': False,
 'seed': 5,
 'n_train': 1000,
 'batch_size': 1000,
 'width': 1000,
 'activation': 'tanh',
 'lr': 0.05,
 'wd': 5e-05,
 'use_cpu': True,
 'add_bc': False,
 'train_mins': [0.0, 0.0, 0.0, 0.0],
 'train_maxes': [0.0, 1.0, 1.0, 1.0],
 'val_mins': [0.0, 0.2, 0.2, 0.2],
 'val_maxes': [0.1, 0.8, 0.8, 0.8],
 'soln': 4,
 'c': 1.0,
 'param_count': 20000,
 'mu_inputs': [0.0,
  0.47192034125328064,
  0.4818880558013916,
  0.5032305717468262],
 'sigma_inputs': [0.0,
  0.298227459192276,
  0.2809867858886719,
  0.28577721118927],
 'mu_targets': [-0.07244887948036194,